In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
from PIL import Image
import json
import os

# Load saved model
model = tf.keras.models.load_model(
    r"C:\Users\Prishu Baranwal\calorie-tracker\models\food_classifier.h5"
)

# Load nutrition data
nutrition_df = pd.read_csv(r"C:\Users\Prishu Baranwal\dataset\nutrition.csv")

print("✅ Model loaded!")
print("✅ Nutrition data loaded!")
print(f"Nutrition columns: {list(nutrition_df.columns)}")

✅ Model loaded!
✅ Nutrition data loaded!
Nutrition columns: ['Food', 'Measure', 'Grams', 'Calories', 'Protein', 'Fat', 'Sat.Fat', 'Fiber', 'Carbs', 'Category']


In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATASET_PATH = r"C:\Users\Prishu Baranwal\dataset\food-101\food-101\images"

# Recreate generator just to get class names
datagen = ImageDataGenerator(rescale=1./255)
temp_gen = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Save class labels to JSON
class_labels = {v: k for k, v in temp_gen.class_indices.items()}
with open(r"C:\Users\Prishu Baranwal\calorie-tracker\models\class_labels.json", "w") as f:
    json.dump(class_labels, f)

print(f"✅ Saved {len(class_labels)} class labels!")
print(f"Sample labels: {list(class_labels.items())[:5]}")

Found 101000 images belonging to 101 classes.
✅ Saved 101 class labels!
Sample labels: [(0, 'apple_pie'), (1, 'baby_back_ribs'), (2, 'baklava'), (3, 'beef_carpaccio'), (4, 'beef_tartare')]


In [4]:
def predict_food(image_path):
    # Load and preprocess image
    img = Image.open(image_path).resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model.predict(img_array)
    top3_indices = np.argsort(predictions[0])[::-1][:3]

    results = []
    for idx in top3_indices:
        food_name = class_labels[idx]
        confidence = predictions[0][idx] * 100
        results.append({"food": food_name, "confidence": f"{confidence:.1f}%"})

    return results

print("✅ Prediction function ready!")

✅ Prediction function ready!


In [5]:
def get_nutrition(food_name, grams=100):
    # Clean food name for matching
    food_search = food_name.replace("_", " ").lower()

    # Search in nutrition dataframe
    nutrition_df['Food'] = nutrition_df['Food'].str.lower()
    match = nutrition_df[nutrition_df['Food'].str.contains(food_search, na=False)]

    if len(match) > 0:
        row = match.iloc[0]
        calories = (float(str(row['Calories']).replace(',','')) / 100) * grams
        protein  = (float(str(row['Protein']).replace(',','').replace('g','')) / 100) * grams

        return {
            "food"    : food_name.replace("_", " ").title(),
            "grams"   : grams,
            "calories": round(calories, 1),
            "protein" : round(protein, 1),
        }
    else:
        return {"food": food_name, "grams": grams, "calories": "Not found", "protein": "Not found"}

print("✅ Nutrition lookup function ready!")

✅ Nutrition lookup function ready!


In [7]:
import os

# Find actual image files inside pizza folder
pizza_path = r"C:\Users\Prishu Baranwal\dataset\food-101\food-101\images\pizza"
images = os.listdir(pizza_path)
print(f"Total pizza images: {len(images)}")
print(f"First 5 images: {images[:5]}")

Total pizza images: 1000
First 5 images: ['1001116.jpg', '1008104.jpg', '1008144.jpg', '1008844.jpg', '1008941.jpg']


In [11]:
# Simulate a full day of eating (using correct food names from CSV)
daily_log = [
    {"food": "pizza",           "grams": 150},
    {"food": "cakes",           "grams": 200},  # fixed from caesar_salad
    {"food": "chocolate fudge", "grams": 100},  # fixed from chocolate_cake
]

total_calories = 0
total_protein  = 0

print("=" * 45)
print("📅 DAILY NUTRITION LOG")
print("=" * 45)

for meal in daily_log:
    n = get_nutrition(meal['food'], meal['grams'])
    if n['calories'] != "Not found":
        total_calories += n['calories']
        total_protein  += n['protein']
    print(f"🍽️  {n['food']} ({n['grams']}g)")
    print(f"    Calories: {n['calories']} kcal | Protein: {n['protein']}g")

print("=" * 45)
print(f"🔥 Total Calories : {round(total_calories, 1)} kcal")
print(f"💪 Total Protein  : {round(total_protein, 1)} g")

# Health risk check
if total_calories > 2500:
    print("⚠️  Health Risk  : High Calorie Intake — Obesity Risk")
elif total_calories < 1200:
    print("⚠️  Health Risk  : Too Low Calories — Deficiency Risk")
else:
    print("✅  Health Risk  : Balanced Diet")

if total_protein < 50:
    print("⚠️  Protein Alert : Low Protein Intake!")
else:
    print("✅  Protein Level : Adequate")
print("=" * 45)

📅 DAILY NUTRITION LOG
🍽️  Pizza (150g)
    Calories: 270.0 kcal | Protein: 12.0g
🍽️  Cakes (200g)
    Calories: 500.0 kcal | Protein: 14.0g
🍽️  Chocolate Fudge (100g)
    Calories: 420.0 kcal | Protein: 5.0g
🔥 Total Calories : 1190.0 kcal
💪 Total Protein  : 31.0 g
⚠️  Health Risk  : Too Low Calories — Deficiency Risk
⚠️  Protein Alert : Low Protein Intake!
